In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.lines as mlines
import regionmask
import glob
import numpy as np
import xarray as xr
import rioxarray
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import AxesGrid
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.crs as ccrs
from pyproj import CRS, Transformer

## Generate a land mask for ERA5

In [ ]:
region_ten = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")
study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")
era5grid = xr.open_dataset("/net/rain/hyclimm/data/projects/SynFire/WP1/ERA5_Variable_Anomaly_by_Weather_Regime/Anomaly_fwi_DJF_AR.nc")

In [ ]:
region_land = gpd.GeoDataFrame({"geometry": [gpd.clip(study_area, geom.geometry).unary_union for _, geom in region_ten.iterrows()],
                                "region": region_ten["region"],
                                "abbrev": region_ten["abbrev"]},
                                crs = "EPSG:4326")

In [ ]:
region_land.to_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Study_Area_32E_10_regions.shp")

In [ ]:
mask_era5 = regionmask.mask_geopandas(region_land, era5grid.lon.values, era5grid.lat.values)

In [ ]:
mask_era5.to_netcdf("/net/rain/hyclimm/data/projects/SynFire/Study_Area/land_mask_ERA5_EPSG4326_study_area_32E.nc")

## Plot fields

In [ ]:
def plot_anomaly_wr(wrname, save):
    
    #-------------------------------
    #layout
    
    fig = plt.figure(figsize = (20, 20))

    #-------------------------------
    #Set up a grid of 5x3 images. Each row has its colorbar.
    grid = AxesGrid(fig, 111,
                    nrows_ncols=(5, 3),
                    axes_pad=0.10,
                    share_all=True,
                    label_mode = "all",
                    cbar_location="right",
                    cbar_mode="edge",
                    cbar_size="5%",
                    cbar_pad="5%",
                    direction = "row"
                   )

    #load study region
    study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")
    regions = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")

    #load land mask
    land_mask_binary = xr.open_dataarray("/net/rain/hyclimm/data/projects/SynFire/Study_Area/land_mask_ERA5_EPSG4326_study_area_32E.nc")


    #-------------------------------
    #colorlist

    #BuRd 10 (for tasmax, sfcWind, fwi)
    colorlist1 = ["#053061", "#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"]
    cmap1 = ListedColormap(colorlist1)
    bounds1 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm1 = BoundaryNorm(bounds1, cmap1.N)

    #RdBu 10 (for tp, hurs)
    colorlist2 = colorlist1[::-1]
    cmap2 = ListedColormap(colorlist2)
    bounds2 = [-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75] #len(bounds) = len(colors) + 1
    norm2 = BoundaryNorm(bounds2, cmap2.N)

    #label
    label_dict = dict(zip(["sfcWind", "pr", "hurs", "tasmax", "fwi"], ["Wind speed", "Total precipitation", "Relative humidity", "Max temperature", "Fire weather index"]))
    sub_lb_dict = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)", "(i)", "(j)", "(k)", "(l)", "(m)", "(n)", "(o)"]

    
    #-------------------------------
    i = 0
    
    for varname in ["tasmax", "hurs", "sfcWind", "pr", "fwi"]:
        
        #-------------------------------
        # specify colormap
        if varname in ["tasmax", "sfcWind", "fwi"]:
            cmap = cmap1
            norm = norm1
            bounds = bounds1
        else:
            cmap = cmap2
            norm = norm2
            bounds = bounds2
            
        
        for season in ["MAM", "JJA", "SON"]:

            #-------------------------------
            #load and reproject anomaly data
            indir = "/net/rain/hyclimm/data/projects/SynFire/WP1/ERA5_Variable_Anomaly_by_Weather_Regime/" 
            var = xr.open_dataarray(os.path.join(indir, f"Anomaly_{varname}_{season}_{wrname}.nc"))

            #mask out ocean
            var = var.where(land_mask_binary.notnull())

            #expand the spatial range a bit for better visual effects (fill value NAN)
            new_lon = np.arange(-12, 34.25, 0.25)
            new_lat = np.arange(72, 32.75, -0.25)
            var_lonlat = var.reindex(lon = new_lon, lat = new_lat)
        
            
            #spatial extent
            lon = var_lonlat.lon.values
            lat = var_lonlat.lat.values
            extent = [lon.min(), lon.max(), lat.min(), lat.max()]
                
                
            #-------------------------------
            im = grid[i].imshow(var_lonlat, extent=extent, norm = norm, cmap = cmap, origin = "upper")
            study_area.boundary.plot(ax = grid[i], edgecolor = "black", linewidth = 0.2)
            regions.boundary.plot(ax = grid[i], edgecolor = "black", linewidth = 0.5)

            #turn off x, y label
            grid[i].tick_params(left = False, labelleft = False, bottom = False, labelbottom = False)

            #add colorbar
            if i%3 == 2:
                grid.cbar_axes[i//3].colorbar(im, extend='both', boundaries=bounds, ticks=bounds).ax.tick_params(labelsize=10)

            #add y label
            if not i%3:
                grid[i].set_ylabel(label_dict[varname], fontsize = 16)

            #add season label
            if varname == "tasmax":
                grid[i].set_title("MAM" if i == 0 else "JJA" if i == 1 else "SON", fontsize = 16)

            #add subpanel label
            grid[i].text(0.03, 0.92, sub_lb_dict[i], transform = grid[i].transAxes, fontsize = 16)
            
            i += 1

    
    for cax in grid.cbar_axes:
        cax.axis[cax.orientation].set_label('Standardized anomaly (-)')
        cax.set_yticks([-0.75, -0.6, -0.45, -0.3, -0.15, 0, 0.15, 0.3, 0.45, 0.6, 0.75])
        cax.set_yticklabels(["-0.75", "-0.60", "-0.45", "-0.30", "-0.15", "0", "0.15", "0.30", "0.45", "0.60", "0.75"])
        cax.axis[cax.orientation].label.set_size(13)
    
    if save:
        plt.savefig(f"/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_{wrname}_atmos_standardized_anomaly.png", dpi = 600, bbox_inches = "tight")

    plt.show()

In [ ]:
for wr in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]:
    plot_anomaly_wr(wr, True)

# Plot averages (latitude weighted)

In [ ]:
#load mask
region_mask = xr.open_dataarray("/net/rain/hyclimm/data/projects/SynFire/Study_Area/land_mask_ERA5_EPSG4326_study_area_32E.nc")
region_mask_key = dict(zip(["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"], [i for i in range(10)]))

In [ ]:
rows = []

for wrname in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"]:

    for season in ['MAM', 'JJA', 'SON']:

        for reg in ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]:
            
            val = dict()
            
            for varname in ['tasmax', 'hurs', 'sfcWind']:
                
                var = xr.open_dataarray(f'/net/rain/hyclimm/data/projects/SynFire/WP1/ERA5_Variable_Anomaly_by_Weather_Regime/Anomaly_{varname}_{season}_{wrname}.nc')
                
                #calculate weight
                
                weights = np.cos(np.deg2rad(var.lat))
                weights.name = 'weights'
                
                
                var_reg = var.where(region_mask == region_mask_key[reg], drop = True)

                #weighted mean
                var_reg_weighted_mean = var_reg.weighted(weights).mean(skipna = True).values
                
                #weighted standard deviation
                var_reg_weighted_variance = (
                    (var_reg - var_reg_weighted_mean)**2
                ).weighted(weights).mean(skipna = True)
                
                var_reg_weighted_std = np.sqrt(var_reg_weighted_variance).values

                

                val.update({f'{varname}_mean': var_reg_weighted_mean, f'{varname}_std': var_reg_weighted_std})


            val.update({'region': reg, 'season': season, 'wr': wrname})
            rows.append(val)

df = pd.DataFrame(rows)

In [ ]:
#save
#df.to_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/ERA5_Variable_Anomaly_by_Weather_Regime/Standardized_anomaly_tasmax_hurs_sfcWind_mean_std_by_region_and_weather_regime_lat_weighted.csv', index = False)

In [ ]:
df = pd.read_csv('/net/rain/hyclimm/data/projects/SynFire/WP1/ERA5_Variable_Anomaly_by_Weather_Regime/Standardized_anomaly_tasmax_hurs_sfcWind_mean_std_by_region_and_weather_regime_lat_weighted.csv')

fig, axes = plt.subplots(3, 3, figsize = (14, 14))
axes = axes.flatten()
axes[-1].axis('off')

#marker for each region
marker_dict = {'IP':'o',
               'EMD':'v',
               'AL':'>',
               'BI':'x',
               'NEA':'d',
               'WMD':'^',
               'FR':'<',
               'SEA': 'P',
               'ME':'D',
               'SC':'s',
              }

#subplot title
title_dict = {'EuBL': '(a) EuBL',
              'ScBL': '(b) ScBL',
              'ZO': '(c) ZO',
              'AT': '(d) AT',
              'ScTr': '(e) ScTr',
              'AR': '(f) AR',
              'GL': '(g) GL',
              'no': '(h) no'}



#highlight
mam_dict = {'EuBL': ['BI', 'SC', 'NEA', 'ME', 'SEA', 'IP'],
            'ScBL': ['BI', 'SC', 'NEA'],
            'ZO': ['IP', 'NEA', 'SEA', 'EMD'],
            'ScTr': ['IP', 'EMD']}

jja_dict = {'ScBL': ['SC', 'WMD', 'EMD', 'SEA', 'NEA'],
            'AR': ['FR', 'IP', 'WMD', 'EMD', 'NEA', 'SC'],
            'AT': ['SEA', 'WMD', 'EMD'],
            'EuBL': ['SC', 'NEA'],
            'ZO': ['SC', 'NEA'],
            'no': ['IP', 'WMD', 'EMD']}

son_dict = {'EuBL': ['IP', 'NEA'],
            'ZO': ['SEA', 'EMD', 'NEA'],
            'ScTr': ['IP', 'WMD', 'EMD']}

#season color
color_dict = {'MAM': '#0d98ba',
              'JJA': '#fe7d6a',  #light pink
              'SON': '#895129'}
 
for wrname, ax in zip(['EuBL', 'ScBL', 'ZO', 'AT',  'ScTr', 'AR', 'GL', 'no'], axes[:-1]):
    
    ax.axvline(x = 0, linestyle = '--', color = '#d3d3d3', zorder = 0)
    ax.axhline(y = 0, linestyle = '--', color = '#d3d3d3', zorder = 0)
    ax.set_aspect('equal')
    
    for season in ['MAM', 'JJA', 'SON']:

        dfsub = df[(df['wr'] == wrname) & (df['season'] == season)]
        season_dict = mam_dict if season == 'MAM' else jja_dict if season == 'JJA' else son_dict

        

        for _, row in dfsub.iterrows():
            reg = row['region']

            #set colors
            if wrname in season_dict.keys() and reg in season_dict[wrname]:
                col = color_dict[season]
                zorder = 2
            else:
                col = '#e0e0e0'
                zorder = 1
                
            
            x = row['tasmax_mean']
            xerr = row['tasmax_std']
            y = row['hurs_mean']
            yerr = row['hurs_std']

            ws = row['sfcWind_mean']
            ls = '-' if ws > 0 else '--'

            ax.plot(x, y, marker = marker_dict[reg], markersize = 8, color = col, zorder = zorder)
            ax.plot([x-xerr, x+xerr], [y, y], linestyle = ls, color = col, zorder = zorder)
            ax.plot([x, x], [y-yerr, y+yerr], linestyle = ls, color = col, zorder = zorder)

    ax.set_xlim(-1, 1)
    ax.set_xticks([-1.0, -0.5, 0, 0.5, 1.0])
    ax.set_xticklabels(["-1.0", "-0.5", "0", "0.5", "1.0"])
    ax.set_ylim(-1, 1)
    ax.set_yticks([-1.0, -0.5, 0, 0.5, 1.0])
    ax.set_yticklabels(["-1.0", "-0.5", "0", "0.5", "1.0"])
    ax.tick_params(axis = "both", labelsize = 12)
    ax.set_xlabel('Max temperature (-)', fontsize = 15, labelpad = 0)
    ax.set_ylabel('Mean relative humidity (-)', fontsize = 15, labelpad = 0)
    ax.text(0.05, 0.9, title_dict[wrname], transform = ax.transAxes, fontsize = 15)

#wind speed legend
x1, y1 = -0.85, -0.1
x2, y2 = 0.15, -0.1

axes[-1].set_xlim(-1, 1)
axes[-1].set_ylim(-1, 1)
axes[-1].plot([x1 - 0.15, x1 + 0.15], [y1, y1], color='black', linestyle='-')  
axes[-1].plot([x1, x1], [y1 - 0.15, y1 + 0.15], color='black', linestyle='-')  
axes[-1].text(-0.7, -0.1, 'Positive', fontsize = 15, va = 'center')

axes[-1].plot([x2 - 0.15,  x2 + 0.15], [y2, y2], color='black', linestyle='--')  
axes[-1].plot([x2, x2], [y2 - 0.15, y2 + 0.15], color='black', linestyle='--')  
axes[-1].text(0.3, -0.1, 'Negative', fontsize = 15, va = 'center')

axes[-1].text(0, 0.1, 'Wind speed', fontsize = 18, ha = 'center')

#other legends
hd_reg = [mlines.Line2D([0], [0], marker=marker_dict[region], color='black',
                       label= 'NEE' if region == 'NEA' else 'SEE' if region == 'SEA' else 'CE' if region == 'ME' else region, linestyle='None', markersize=8) for region in marker_dict.keys()]

hd_season = [mlines.Line2D([0], [0], marker='s', color=cl,
                       label=lb, linestyle='None', markersize=8) for (cl, lb) in [(color_dict['MAM'], 'MAM'), (color_dict['JJA'], 'JJA'), (color_dict['SON'], 'SON'), ('#e0e0e0', 'Not Shown')]]

lg_reg = axes[-1].legend(handles = hd_reg, loc = 'lower center', frameon = False, ncol = 5, bbox_to_anchor = (0.5, 0), handletextpad = -0.5, 
           fontsize = 13, columnspacing = 0.5, title = 'Region', title_fontsize = 18)

lg_season = axes[-1].legend(handles=hd_season, title='Season', loc='lower center', bbox_to_anchor = (0.45, 0.65), ncol = 2, frameon = False, fontsize = 15, title_fontsize = 18,
           handletextpad = -0.5, columnspacing = 3.5)

axes[-1].add_artist(lg_reg)

plt.subplots_adjust(wspace = 0.28)

plt.savefig('/home/lixinh/SynFire_Scripts/WP1/Figures/v5_5_weather_regime_seasonal_anomaly.png', dpi = 600, bbox_inches = 'tight')